# 3DXRD indexing notebook - Grid indexing with tolerances from the data
__Written by Haixing Fang, Jon Wright and James Ball__
__Date: 19/11/2026__

The same route as `1_index_grid.ipynb` (strong ring peaks, grid search, makemap), with the tolerances as there
(`cf_strong_dstol`, `gridpars['DSTOL']`, `makemap_tol_seq`), but the peak-count cuts are measured from the data:

| stock parameter | here |
|---|---|
| `frac` / `minpeaks` (`NPKS`) | from random orientations scored like candidates at a few grid points |
| `absolute_minpks` | each candidate is scored on **all** peaks; it is kept if it explains more unclaimed peaks than random lattices do, so that about `max_false_grains` false grains are expected per pass |

The grid search is repeated on the peaks no accepted grain claims (`max_passes`): in a crowded sample the indexer
fills up with candidates at each grid point and misses grains.

**Several phases**: run the notebook once per phase. Each run adds a column `labels_<phase_str>` (grain id, -1 = not
this phase) to the 3D peaks. Peaks claimed in another phase's column are not used (see `exclude_phases`), so the phases
claim peaks in the order you run them. The columns `labels`, `h`, `k`, `l`, `drlv2`, ... are written as in the stock
notebook and belong to the phase run last.

In [ ]:
import os

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())

In [ ]:
# this cell is tagged with 'parameters'
# to view the tag, select the cell, then find the settings gear icon (right or left sidebar) and look for Cell Tags

# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

# desination of H5 files
# replace below with e.g.:
# dset_path = '/data/visitor/expt1234/20240101/PROCESSED_DATA/sample/dataset/sample_dataset.h5'

dset_path = ''

phase_str = 'Fe'

# path to parameters .json/.par
parfile = ''

# peak filtration options
cf_strong_frac = 0.999   # intensity fraction of the peaks used for the grid search and the first refinements
cf_strong_dsmax = 1.017  # highest d* for the grid search peaks
cf_strong_dstol = 0.025  # d* window around each ring

# indexing options
rings_to_use = [0, 1, 3]  # rings for the grid search: clear of other phases and of each other

# multi-phase: peaks claimed by these phases (columns labels_<phase>) are not used
# None: every other phase that has a column; []: use all peaks
exclude_phases = None

# makemap options
symmetry = "cubic"
makemap_tol_seq = [0.02, 0.015, 0.01]  # the last one is the hkl tolerance for acceptance and the final refinement

gridpars = {
        'DSTOL' : 0.004,
        'RING1'  : [1,0,],
        'RING2' : [0,],
        'NUL' : True,
        'FITPOS' : True,
        'tolangle' : 0.50,
        'toldist' : 100.,
        'NTHREAD' : 1 ,
}

grid_xlim = 600  # SAME UNITS AS DISTANCE IN PAR FILE - extent away from rotation axis to search for grains
grid_ylim = 600
grid_zlim = 200
grid_step = 100  # step size of search grid, SAME UNITS AS DISTANCE IN PAR FILE

# expected number of false grains allowed in each pass
max_false_grains = 1
# grid searches on the peaks left over (stops earlier when a pass finds no grain)
max_passes = 3

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import os
import sys
import time
import numpy as np
from matplotlib import pyplot as plt
import h5py

import random
import ImageD11.cImageD11
import ImageD11.grain
import ImageD11.indexing
import ImageD11.columnfile
from ImageD11.sinograms import dataset
from ImageD11.peakselect import sorted_peak_intensity_mask, rings_mask
from ImageD11.nbGui import nb_utils as utils
from ImageD11.grid_index_parallel import grid_index_parallel
from ImageD11 import grid_stats

%matplotlib widget

# Load data
## Dataset

In [ ]:
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

## Parameters
Specify the path to your parameter file.

You can optionally set up some default parameters for either an Eiger or Frelon detector like so:
```python
from ImageD11.parameters import AnalysisSchema
asc = AnalysisSchema.from_default(detector='eiger')  # or detector='frelon'
asc.save('./pars.json')
```
Please note in this case that you will still have to update the `geometry.par` values accordingly for your experiment.  
If you haven't already, you should run one of the calibration notebooks to determine these.

In [ ]:
ds.parfile = parfile
ds.save()

## Phases
If the parameter file was a json, we can access the unit cells via `ds.phases.unitcells`

In [ ]:
ds.phases = ds.get_phases_from_disk()
ds.phases.unitcells

In [ ]:
ucell = ds.phases.unitcells[phase_str]
print(ucell)

## Peaks
With several phases, the peaks claimed by the other phases (`labels_<phase> >= 0`) are left out: `cf_phase` holds the
rows of `cf_3d` this phase can use (`phase_rows`).

In [ ]:
cf_3d = ds.get_cf_3d_from_disk()
ds.update_colfile_pars(cf_3d, phase_name=phase_str)
label_col = 'labels_' + phase_str
others = [t[len('labels_'):] for t in cf_3d.titles if t.startswith('labels_') and t != label_col]
if exclude_phases is not None:
    others = [p for p in others if p in exclude_phases]
free = np.ones(cf_3d.nrows, bool)
for p in others:
    free &= cf_3d['labels_' + p] < 0
    print('phase', p, 'claims', (cf_3d['labels_' + p] >= 0).sum(), 'peaks')
phase_rows = np.flatnonzero(free)
cf_phase = cf_3d.copyrows(phase_rows)
print(cf_phase.nrows, 'of', cf_3d.nrows, 'peaks available for', phase_str)

# Visualise data
### $d^{*}$ vs $\eta$

In [ ]:
skip = 1  # we can skip peaks to speed up plotting if needed
ucell.makerings(cf_phase.ds.max())
fig, ax = plt.subplots(figsize=(10,5), layout='constrained')
ax.plot(cf_phase.ds[::skip], cf_phase.eta[::skip], ',')
ax.vlines(ucell.ringds, -50, 50, color='red')
ax.set(xlabel=r'$d^{*}~(\AA^{-1})$', ylabel=r'$\eta~(\degree)$', title='2D azimuthal transform')
plt.show()

# Peak selection
As in the stock notebook: peaks within `cf_strong_dstol` of a ring, the strongest `cf_strong_frac` of them.
* `grid_peaks.flt`: those up to `cf_strong_dsmax` on `rings_to_use`, for the grid search.
* the same selection on all rings, to refine the grain positions after the grid search.
* all peaks this phase can use (`cf_phase`), for the acceptance step and the final refinement.

In [ ]:
geom = grid_stats.PeakGeometry(cf_phase)  # scores candidates on all peaks, below

def strong_rows(dsmax):
    # rows of cf_phase that select_ring_peaks_by_intensity would keep
    rows = np.flatnonzero(rings_mask(cf_phase, cf_strong_dstol, dsmax, ucell))
    return rows[sorted_peak_intensity_mask(cf_phase.copyrows(rows), frac=cf_strong_frac, doplot=0.5)]

strong = strong_rows(cf_strong_dsmax)
indexer = ImageD11.indexing.indexer_from_colfile_and_ucell(cf_phase.copyrows(strong), ucell)
indexer.ds_tol = cf_strong_dstol
indexer.assigntorings()
grid_rows = strong[np.isin(indexer.ra, rings_to_use)]
allrings_rows = strong_rows(cf_phase.ds.max())

cf_strong = cf_phase.copyrows(grid_rows)
grid_peaks_path = 'grid_peaks.flt'
cf_phase_path = 'cf_phase.flt'
cf_strong.writefile(grid_peaks_path)
print(cf_strong.nrows, 'grid peaks;', len(allrings_rows), 'strong peaks on all rings;', cf_phase.nrows, 'peaks in all')

skip = 1  # we can skip peaks to speed up plotting if needed
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.plot(cf_phase.ds[::skip], cf_phase.sum_intensity[::skip],',', label='cf_phase')
ax.plot(cf_strong.ds[::skip], cf_strong.sum_intensity[::skip],',', label='grid peaks')
ax.vlines(ucell.ringds, 1e3, 1e4, color='red')
ax.set(xlabel=r'$d^{*}~(\AA^{-1})$', ylabel='Intensity', yscale='log', title='Peak filtration')
ax.legend()
plt.show()

# Grid index
## Cuts for the grid search from random orientations
Random orientations at three grid points, scored exactly as the grid indexer scores its candidates (only the peaks
within `DSTOL` of a ring at that point, `DSFILTER`):

* `NPKS_INDEX` (an indexing candidate is refined): above the 99th percentile of the scores at `makemap_tol_seq[0]`;
* `NPKS` (a refined candidate is kept): above the 99.9th percentile after `domap`.

These cuts only remove most chance lattices before the expensive steps. The grains are chosen later, on all peaks.
Peaks expected per grain for the rings used (360 degree scan: $2\sum m_r$) is printed for comparison.

In [ ]:
oldparfile = phase_str + '.par'
ds.phases.to_old_pars_file(oldparfile, phase_str)  # export parameter file
nproc = max( ImageD11.cImageD11.cores_available() - 1 , 1)
omega_slop = ds.ostep/2

gridpars['DSFILTER'] = True
gridpars['COSTOL'] = np.cos(np.radians(90 - ds.ostep))
gridpars['NPROC'] = nproc
gridpars['NTHREAD'] = 1
gridpars['OMEGAFLOAT'] = omega_slop
gridpars['TOLSEQ'] = makemap_tol_seq
gridpars['SYMMETRY'] = symmetry

translations = [(t_x, t_y, t_z) # grid to search
    for t_x in range(-grid_xlim, grid_xlim+1, grid_step)
    for t_y in range(-grid_ylim, grid_ylim+1, grid_step)
    for t_z in range(-grid_zlim, grid_zlim+1, grid_step) ]

random.seed(42) # reproducible
random.shuffle(translations)

peaks_expected = 2 * sum(len(ucell.ringhkls[ucell.ringds[r]]) for r in rings_to_use)
print(len(translations), 'grid points; peaks expected per grain on rings', rings_to_use, ':', peaks_expected)

def grid_cuts(fltfile, seed):
    """ NPKS_INDEX, NPKS from random orientations at three grid points """
    rng = np.random.RandomState(seed)
    points = [translations[i] for i in rng.choice(len(translations), min(3, len(translations)), replace=False)]
    s_index, s_map = grid_stats.grid_null_scores(fltfile, oldparfile, gridpars, points, 1000, nproc, seed)
    npks_index, npks = int(np.percentile(s_index, 99)), int(np.percentile(s_map, 99.9))
    print('random orientations: index score median %d, 99%% %d -> NPKS_INDEX %d;  after domap median %d, 99.9%% %d, max %d -> NPKS %d'
          % (np.median(s_index), npks_index, npks_index, np.median(s_map), npks, s_map.max(), npks))
    return npks_index, npks, s_index, s_map

## Grid search, then refinement
`grid_pass` runs the grid search on a peak file, then makemap (`grid_stats.refine_grains`, the same refinement
in-process) on the strong peaks of all rings with `makemap_tol_seq[-1]`, repeated while the median position change
still halves (one makemap leaves grains several grid steps' worth of um short of their positions). Candidates are
refined in parallel chunks; peaks are shared between chunks until the acceptance step settles them. A grain whose fit
fails (a left handed UBI from too few peaks) is dropped; `makemap.py` would stop on it.

In [ ]:
tmpfiles = [grid_peaks_path, cf_phase_path + '.new']

def converge(grains, colfile, tol, tag, parallel, maxiter=6):
    """ makemap until the median position change stops halving.
    parallel: candidates, refined in chunks (peaks assigned within a chunk; acceptance settles shared peaks).
    Otherwise all grains together, and colfile gets the makemap columns. """
    prev = grains
    last = np.inf
    for i in range(maxiter):
        if parallel:
            cur, kept = grid_stats.refine_grains_parallel(oldparfile, colfile, prev, tol, omega_slop, symmetry, nproc)
        else:
            cur, kept = grid_stats.refine_grains(oldparfile, colfile, prev, tol, omega_slop, symmetry)
        shift = np.median([np.linalg.norm(np.subtract(prev[j].translation, g.translation)) for j, g in zip(kept, cur)])
        print('  makemap %s tol %g: median position change %.3g' % (tag, tol, shift))
        if shift > last / 2:
            break
        prev, last = cur, shift
    return [g for g in cur if 'no peaks' not in g.intensity_info]

def grid_pass(tag, grid_flt, allrings_cf, seed):
    npks_index, npks, s_index, s_map = grid_cuts(grid_flt, seed)
    gp = dict(gridpars, NPKS=npks, NPKS_INDEX=npks_index, output_filename=tag + '_grid.map')
    if os.path.exists(gp['output_filename']):
        os.remove(gp['output_filename'])
    t0 = time.time()
    grid_index_parallel(grid_flt, oldparfile, tag + '_tmp', gp, translations)
    tmpfiles.extend([gp['output_filename'], tag + '_tmp.flt'])
    print('grid search %.0f s' % (time.time() - t0))
    cands = ImageD11.grain.read_grain_file(gp['output_filename'])
    if len(cands) == 0:
        return [], (s_index, s_map, npks)
    return converge(cands, allrings_cf, makemap_tol_seq[-1], tag, parallel=True), (s_index, s_map, npks)

cands1, (s_index, s_map, npks) = grid_pass('pass1', grid_peaks_path, cf_phase.copyrows(allrings_rows), seed=1)
print(len(cands1), 'candidates after pass 1')

fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
bins = np.arange(0, max(s_map.max(), max(float(g.npks) for g in cands1) if cands1 else 0) + 2)
ax.hist(s_map, bins, histtype='step', label='random orientations after domap')
ax.hist([float(g.npks) for g in cands1], bins, histtype='step', label='candidates after makemap (all rings)')
ax.axvline(npks, color='r')
ax.set(yscale='log', xlabel='npks', title='grid search cut NPKS')
ax.legend()
plt.show()

In [ ]:
grains2 = [ImageD11.grain.grain(g.ubi, g.translation) for g in cands1]
for g in grains2:
    g.ref_unitcell = ucell
utils.get_rgbs_for_grains(grains2)
utils.plot_all_ipfs(grains2)

In [ ]:
utils.plot_grain_positions(cands1, colour='npks', centre_plot=False, size_scaling=0.5)

# Accepting grains on all peaks
**Score.** Each candidate indexes the peaks within `makemap_tol_seq[-1]` of an integer hkl, with g-vectors computed for its own
position, on **all** peaks of the phase (the strong-peak files are too sparse to tell weak grains from chance).

**Unique peaks.** Candidates are taken strongest first. A candidate is accepted if more than $c$ of its peaks are not
claimed by an accepted grain, and then claims them. A duplicate of an accepted grain, or a lattice that re-uses its
peaks, has few unclaimed peaks and is rejected.

**Cut.** Random orientations are placed at candidate positions, refined through `makemap_tol_seq` like a candidate
and counted the same way. With $N$ candidates, about $N\,P(u^\mathrm{rand} > c)$ false grains are expected; $c$ is
the smallest value for which this is at most `max_false_grains`. $P$ is read off the random sample (at least
$5N/$`max_false_grains` lattices), so no distribution is assumed.

In [ ]:
final_tol = makemap_tol_seq[-1]
claimed = np.zeros(cf_phase.nrows, bool)
accepted = []
expected_false = []

def accept(cands, seed):
    global claimed
    t0 = time.time()
    matched = grid_stats.match_grains(geom, cands, final_tol, nproc)
    n_null = max(2000, int(np.ceil(5 * len(cands) / max_false_grains)))
    rng = np.random.RandomState(seed)
    npos = int(np.ceil(n_null / 10))
    pos = [cands[i].translation for i in rng.choice(len(cands), npos, replace=npos > len(cands))]
    nulls = grid_stats.null_matches(geom, pos, ucell.B, makemap_tol_seq, final_tol, 10, nproc, seed)
    acc, claimed, cut, efalse, unclaimed, null_un = grid_stats.accept_grains(matched, nulls, claimed, max_false_grains)
    print('%d candidates, %d random lattices (unclaimed median %d, max %d) -> cut %d: accepted %d, expected false %.2f; '
          '%d of %d peaks claimed (%.0f s)' % (len(cands), len(nulls), np.median(null_un), null_un.max(), cut, len(acc),
                                              efalse, claimed.sum(), len(claimed), time.time() - t0))
    fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
    b = np.arange(0, max(unclaimed.max(), null_un.max()) + 10, 5)
    ax.hist(unclaimed, bins=b, histtype='step', label='candidates')
    ax.hist(null_un, bins=b, histtype='step', weights=np.full(len(null_un), len(cands) / len(null_un)), label='random lattices (scaled)')
    ax.axvline(cut, c='r')
    ax.set(yscale='log', xlabel='unclaimed peaks (hkl_tol %.4f)' % final_tol)
    ax.legend()
    plt.show()
    expected_false.append(efalse)
    return [cands[j] for j in sorted(acc)]

accepted += accept(cands1, seed=1)

# More passes on the peaks left over
The grid peaks and strong peaks that no accepted grain claims are indexed again, until a pass accepts no grain or
`max_passes` is reached. Duplicates of earlier grains have few unclaimed peaks and are rejected as above.

In [ ]:
for npass in range(2, max_passes + 1):
    tag = 'pass%d' % npass
    grid_flt = tag + '_grid_peaks.flt'
    cf_phase.copyrows(grid_rows[~claimed[grid_rows]]).writefile(grid_flt)
    tmpfiles.append(grid_flt)
    cands, _ = grid_pass(tag, grid_flt, cf_phase.copyrows(allrings_rows[~claimed[allrings_rows]]), seed=npass)
    new = accept(cands, seed=npass) if len(cands) else []
    print('pass %d: %d new grains' % (npass, len(new)))
    accepted += new
    if len(new) == 0:
        break
print(len(accepted), 'grains accepted, expected false %.2f (sum over passes)' % sum(expected_false))

In [ ]:
utils.plot_grain_positions(accepted, colour='npks', centre_plot=False, size_scaling=0.5)

# Grain refinement on all peaks
All accepted grains are refined together against all peaks of the phase with `makemap_tol_seq[-1]` (each peak goes to the grain
that fits it best), repeated while the positions still move.

In [ ]:
mm = cf_phase.copy()
grains_final = converge([ImageD11.grain.grain(g.ubi, g.translation) for g in accepted], mm, final_tol, 'final', parallel=False)
ImageD11.grain.write_grain_file('final.map', grains_final)
mm.writefile(cf_phase_path + '.new')
tmpfiles.append('final.map')
for g in grains_final:
    g.ref_unitcell = ucell
utils.get_rgbs_for_grains(grains_final)
print(f"{len(grains_final)} final grains")

## Peak labels
The makemap columns (`labels`, `drlv2`, `h`, `k`, `l`, ...) are put back into `cf_3d`: rows of this phase from
makemap, other rows as unassigned (`labels` -1). `labels_<phase_str>` is a copy of `labels` that later runs for other
phases keep.

In [ ]:
assert mm.nrows == len(phase_rows)
per_grain = {'labels': -1, 'drlv2': 1, 'tth_per_grain': -1, 'eta_per_grain': 0, 'omegacalc_per_grain': 0,
             'h': 0, 'k': 0, 'l': 0, 'hr': 0, 'kr': 0, 'lr': 0}
for t in mm.titles:
    if t in per_grain or t not in cf_3d.titles:
        col = np.full(cf_3d.nrows, per_grain.get(t, 0), dtype=mm[t].dtype)
    else:
        col = cf_3d[t].copy()
    col[phase_rows] = mm[t]
    cf_3d.addcolumn(col, t)
cf_3d.addcolumn(cf_3d.labels.astype(np.int32), label_col)
print((cf_3d[label_col] >= 0).sum(), 'peaks assigned to', len(grains_final), phase_str, 'grains')

In [ ]:
utils.plot_grain_positions(grains_final, colour='npks', centre_plot=False, size_scaling=0.5)

In [ ]:
utils.plot_grain_positions(grains_final, colour='z', centre_plot=False, size_scaling=0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7), layout='constrained')
ax.hist([float(grain.npks) for grain in grains_final], bins=30)
ax.set(xlabel='Number of peaks per grain', ylabel='Count', title='Histogram of peaks per grain')
plt.show()

In [ ]:
utils.plot_grain_histograms(cf_phase_path + '.new', 'final.map', oldparfile, omega_slop, tol=final_tol)

In [ ]:
unit_cell_lengths = np.array([np.sort(g.unitcell.copy()) for g in grains_final])
median_unit_cell = np.median(unit_cell_lengths, axis=0)
print("Median unit cell:", median_unit_cell)
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(unit_cell_lengths[:,:3])
ax.hlines(median_unit_cell[:3], 0, len(unit_cell_lengths))
ax.set(xlabel="Grain ID", ylabel="Unit cell length")
plt.show()

# Export data
The peaks go to group `peaks` of the 3D peaks file (as in the stock notebook, plus `labels_<phase_str>` as an integer
column), the grains to group `phase_str` of the grains file (replaced if it exists).

In [ ]:
ImageD11.columnfile.colfile_to_hdf(cf_3d, ds.col3dfile, name='peaks')
with h5py.File(ds.col3dfile, 'a') as hout:  # colfile_to_hdf writes columns it does not know as float
    grp = hout['peaks']
    if label_col in grp and grp[label_col].dtype != np.int32:
        del grp[label_col]
    if label_col in grp:
        grp[label_col][:] = cf_3d[label_col]
    else:
        grp.create_dataset(label_col, data=cf_3d[label_col].astype(np.int32))
if os.path.exists(ds.grainsfile):
    with h5py.File(ds.grainsfile, 'a') as hout:  # write_grain_file_h5 cannot overwrite a group
        if phase_str in hout:
            del hout[phase_str]
ds.save_grains_to_disk(grains_final, phase_name=phase_str)

In [ ]:
ds.save()

# Deleting temporary files

In [ ]:
for path in set(tmpfiles):
    if os.path.exists(path):
        os.remove(path)